<a href="https://colab.research.google.com/github/AshrfCode/Anan-Tirgulem/blob/main/tutorial2/tirgul2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
from google.colab import drive
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Connect to Google Drive
drive.mount('/content/drive')

# File Path for students.json file
FILE_PATH = '/content/drive/MyDrive/Anan/tirgul2/students.json'

# Function that loads the data
def load_data():
    if not os.path.exists(FILE_PATH):
        print(f"File not found at path: {FILE_PATH}")
        return []
    with open(FILE_PATH, 'r', encoding='utf-8') as f:
        return json.load(f)

students_data = load_data()

if not students_data:
    print("No data found to display. Please ensure the file exists and contains valid information.")
else:
    # --- Styling and Title ---
    title = widgets.HTML("<h2 style='color: #2E86C1; text-align: left;'>🎓 Student Management System - Braude</h2>")

    # --- Widget Creation ---
    # Create a list for the dropdown (displays full name, keeps email as a unique identifier)
    options = [(f"{s.get('first_name', '')} {s.get('last_name', '')}", s.get('email', '')) for s in students_data]

    dropdown = widgets.Dropdown(options=options, description='Select Student:', style={'description_width': 'initial'})

    # Read-only text boxes for student details
    fname_text = widgets.Text(description='First Name:', disabled=True)
    lname_text = widgets.Text(description='Last Name:', disabled=True)
    email_text = widgets.Text(description='Braude Email:', disabled=True)
    courses_text = widgets.Text(description='Courses:', disabled=True)

    # Text box for editing the favorite link
    fav_link_text = widgets.Text(description='Favorite Link:', placeholder='Enter GitHub / LinkedIn link...')

    # Update button (styled)
    update_btn = widgets.Button(
        description='Save Update',
        button_style='success', # Applies green color
        icon='check'
    )

    out = widgets.Output()

    # --- Logic Functions ---
    def populate_form(email):
        """Function that populates the form based on the selected student"""
        for s in students_data:
            if s.get('email') == email:
                fname_text.value = s.get('first_name', '')
                lname_text.value = s.get('last_name', '')
                email_text.value = s.get('email', '')

                # Convert the courses list to a string
                courses = s.get('courses', [])
                courses_text.value = ", ".join(courses) if isinstance(courses, list) else str(courses)

                # Load the favorite link
                fav_link_text.value = s.get('favorite_link', '')
                break

    def on_dropdown_change(change):
        """Listener for dropdown selection changes"""
        if change['type'] == 'change' and change['name'] == 'value':
            with out:
                clear_output()
            populate_form(change['new'])

    def on_update_click(b):
        """Listener for the update button click"""
        with out:
            clear_output()
            selected_email = dropdown.value
            for s in students_data:
                if s.get('email') == selected_email:
                    # Update the information in our data structure
                    s['favorite_link'] = fav_link_text.value
                    break

            # Save back to the file in Drive
            try:
                with open(FILE_PATH, 'w', encoding='utf-8') as f:
                    # ensure_ascii=False is good practice even in English if special characters are ever used
                    json.dump(students_data, f, ensure_ascii=False, indent=4)
                print(f"✅ The favorite link was successfully updated for {fname_text.value} {lname_text.value} and saved to Drive!")
            except Exception as e:
                print(f"❌ Error saving file: {e}")

    # Connect functions to events
    dropdown.observe(on_dropdown_change)
    update_btn.on_click(on_update_click)

    # Initialize the form with the first student in the list
    if options:
        populate_form(options[0][1])

    # --- Layout Configuration ---
    # Arrange the boxes aesthetically
    form_items = [
        dropdown,
        widgets.HTML("<hr>"), # Separator line
        fname_text,
        lname_text,
        email_text,
        courses_text,
        widgets.HTML("<hr>"), # Another separator line
        fav_link_text,
        update_btn,
        out
    ]

    # Use VBox (Vertical Box) to stack the elements
    form = widgets.VBox(form_items, layout=widgets.Layout(
        display='flex',
        flex_flow='column',
        align_items='flex-start', # Left alignment for English text
        width='50%',
        padding='20px',
        border='1px solid #ccc'
    ))

    # Final display of the system
    display(title)
    display(form)